# Anggota 2 — Pemrosesan Pertanyaan

Rionaldi Suhandafa · 245150200111045

# 1. Aturan Pemrosesan

In [1]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd

VERSI_PREPROCESS = "1.0"


STOPWORDS = frozenset("""
yang dan atau dengan adalah merupakan sebagai pada di ke oleh ini itu tersebut
untuk saya aku tolong mohon jelaskan sebutkan dimaksud maksud
apa apakah siapakah siapa kapan kapankah dimana kemana darimana
mana manakah berapa berapakah bagaimana bagaimanakah mengapa kenapa saja
""".split())


ALIASES = (
    (r"\bindeks prestasi kumulatif\b|\bip kumulatif\b", "ipk"),
    (r"\bindeks prestasi\b", "ip"),
    (r"\bsatuan kredit semester\b|\bsistem kredit semester\b", "sks"),
    (r"\bujian akhir semester\b", "uas"),
    (r"\bujian tengah semester\b", "uts"),
    (r"\bprogram studi\b", "prodi"),
)


POLA_TOKEN = re.compile(
    r"\d+(?:,\d+)?|[^\W\d_]+(?:[-'][^\W\d_]+)*(?:\+)?|[≥≤<>=%×/+*−-]",
    re.UNICODE,
)


def normalisasi_teks(teks: str) -> str:

    if not isinstance(teks, str):
        raise TypeError("Teks harus berupa str.")
    hasil = unicodedata.normalize("NFKC", teks).casefold()
    hasil = hasil.replace(">=", "≥").replace("<=", "≤")

    hasil = re.sub(r"(?<=\d)\.(?=\d)", ",", hasil)
    return " ".join(hasil.split())


def tokenisasi(teks: str) -> list[str]:

    hasil = normalisasi_teks(teks)
    for pola, pengganti in ALIASES:
        hasil = re.sub(pola, pengganti, hasil)
    return POLA_TOKEN.findall(hasil)


def preprocess(teks: str) -> list[str]:

    return [token for token in tokenisasi(teks) if token not in STOPWORDS]


def teks_index(passage: dict) -> str:

    return " ".join(
        passage.get(kolom, "")
        for kolom in ("judul_subbab", "konteks", "teks")
        if passage.get(kolom, "")
    )


def preprocess_passage(passage: dict) -> list[str]:
    return preprocess(teks_index(passage))

POLA_TANYA = re.compile(
    r"\b(?:berapa(?:kah)?|siapa(?:kah)?|kapan(?:kah)?|"
    r"(?:di|ke|dari)\s*mana|bagaimana(?:kah)?|mengapa|kenapa|"
    r"apakah|apa|mana(?:kah)?)\b"
)


def jenis_pertanyaan(teks: str) -> tuple[str, str, list[str]]:

    penanda = [m.group() for m in POLA_TANYA.finditer(teks)]
    if not penanda:
        return "TIDAK_DIKENALI", "", []
    pertama = penanda[0]
    if re.fullmatch(r"(?:di|ke|dari)\s*mana", pertama):
        jenis = "DI_MANA"
    elif pertama in {"mengapa", "kenapa"}:
        jenis = "MENGAPA"
    elif pertama == "apakah":
        jenis = "YA_TIDAK"
    else:
        jenis = pertama.removesuffix("kah").upper()
    return jenis, pertama, penanda


def tipe_jawaban(teks: str, jenis: str, penanda: str) -> tuple[str, str]:

    if jenis == "YA_TIDAK":
        return "YA_TIDAK", "pertanyaan konfirmasi"

    if jenis == "SIAPA":
        if re.search(r"\bnama(?:nya)?\b", teks):
            return "ORANG", "siapa disertai permintaan nama"
        return "JABATAN", "siapa tanpa permintaan nama; asumsi domain akademik"
    if jenis == "KAPAN":
        return "WAKTU", "kapan: tanggal, tahun, atau peristiwa awal/akhir"
    if jenis == "DI_MANA":
        return "LOKASI", "penanda lokasi"
    if jenis == "MENGAPA":
        return "ALASAN", "penanda sebab"
    if jenis in {"TIDAK_DIKENALI", "MANA"}:
        return "TIDAK_DIKENALI", "bentuk belum dicakup aturan"


    if jenis == "BERAPA":
        setelah = teks.split(penanda, 1)[1].strip()
        if re.search(r"\b(?:persen|persentase)\b|%", setelah):
            return "PERSENTASE", "berapa disertai satuan persen"
        kepala = re.sub(r"^(?:(?:jumlah|total|minimal|maksimal|minimum|maksimum)\s+)+", "", setelah)
        if re.match(r"(?:sks|satuan kredit|nilai|ipk|ip|indeks prestasi|bobot|skor|orang|kali)\b", kepala):
            return "ANGKA", "objek hitungan numerik"
        if re.match(r"(?:lama|durasi|waktu|menit|jam|hari|minggu|bulan|tahun|semester)\b", kepala):
            return "DURASI", "objek hitungan berupa lama waktu"
        return "ANGKA", "berapa: jumlah numerik sebagai default"

    if re.search(r"\b(?:rumus|menghitung|hitung|perhitungan)\b", teks):
        return "RUMUS", "permintaan rumus atau cara menghitung"
    if re.search(r"\b(?:syarat|syarat-syarat|persyaratan)\b", teks):
        return "SYARAT", "permintaan persyaratan"
    if re.search(r"\b(?:prosedur|mekanisme|langkah|cara)\b", teks) or jenis == "BAGAIMANA":
        return "PROSEDUR", "permintaan cara atau tahapan"
    if re.search(r"\b(?:apa itu|apa yang dimaksud|pengertian|definisi|arti|kepanjangan)\b", teks):
        return "DEFINISI", "permintaan makna istilah"
    if re.search(r"\b(?:persen|persentase)\b|%", teks):
        return "PERSENTASE", "permintaan persentase"
    if re.search(r"\b(?:bobot|rentang|skor|ipk|indeks prestasi kumulatif)\b", teks):
        return "ANGKA", "target besaran numerik"
    if re.search(r"\b(?:nilai huruf|huruf mutu)\b", teks):
        return "NILAI_HURUF", "target konversi ke huruf"
    if re.search(r"\b(?:durasi|lama|batas waktu)\b", teks):
        return "DURASI", "target lama waktu"
    if re.search(r"\bapa saja\b", teks):
        return "DAFTAR", "permintaan daftar"
    if jenis == "APA":
        return "DEFINISI", "apa tanpa target khusus; asumsi definisi"
    return "TIDAK_DIKENALI", "target belum dicakup aturan"


def proses_pertanyaan(pertanyaan: str) -> dict:

    normal = normalisasi_teks(pertanyaan)
    if not normal:
        raise ValueError("Pertanyaan tidak boleh kosong.")
    jenis, penanda, semua_penanda = jenis_pertanyaan(normal)
    tipe, aturan = tipe_jawaban(normal, jenis, penanda)
    tokens = preprocess(pertanyaan)
    kata_kunci = list(dict.fromkeys(tokens))
    catatan = []
    perlu_klarifikasi = tipe == "TIDAK_DIKENALI" or not kata_kunci
    if len(semua_penanda) > 1:
        perlu_klarifikasi = True
        catatan.append("Terdapat beberapa penanda tanya; pecah menjadi satu pertanyaan per masukan.")
    if jenis == "SIAPA":
        catatan.append("Korpus terutama memuat jabatan. Pemetaan SIAPA ke JABATAN adalah heuristik, bukan NER.")
    if tipe == "ORANG":
        catatan.append("Nama orang sangat terbatas; keberadaan jawaban harus diperiksa saat retrieval.")
    if tipe == "WAKTU":
        catatan.append("Kalender akademik tidak tersedia; fakta tahun atau peristiwa mungkin tersedia.")
    if tipe in {"ALASAN", "PROSEDUR", "DAFTAR", "SYARAT", "YA_TIDAK"}:
        catatan.append("Tipe ini memerlukan penanganan anggota 4 dan dapat melampaui jawaban factoid singkat.")
    if re.search(r"\b(?:terbaru|sekarang|saat ini)\b", normal):
        catatan.append("Korpus historis (FILKOM 2020, tata tertib 2016, penilaian 2015); bukan aturan terkini.")
    if perlu_klarifikasi and len(semua_penanda) <= 1:
        catatan.append("Gunakan kata tanya dan sebutkan objek akademik yang ingin ditanyakan.")
    return {
        "pertanyaan": pertanyaan,
        "normalisasi": normal,
        "token": tokenisasi(pertanyaan),
        "token_pencarian": tokens,
        "kata_kunci": kata_kunci,

        "query": " ".join(tokens),
        "jenis_pertanyaan": jenis,
        "penanda_tanya": penanda,
        "tipe_jawaban": tipe,
        "aturan_tipe": aturan,
        "status": "PERLU_KLARIFIKASI" if perlu_klarifikasi else "SIAP_RETRIEVAL",
        "catatan": catatan,
        "versi_preprocess": VERSI_PREPROCESS,
    }

Aturan mengenali jenis pertanyaan, memperkirakan tipe jawaban, dan menyiapkan token untuk pencarian.

# 2. Soal dan Token Passage

In [2]:
with open("../Anggota1_Korpus/output/passages.jsonl", encoding="utf-8") as file:
    passages = [json.loads(line) for line in file]
with open("../Anggota5_Evaluasi_Retrieval/data/pertanyaan_uji.json", encoding="utf-8") as file:
    pertanyaan = json.load(file)

passage_by_id = {p["id"]: p for p in passages}
assert len({soal["id_pertanyaan"] for soal in pertanyaan}) == len(pertanyaan)

for soal in pertanyaan:
    teks_bukti = {
        pid: f'{passage_by_id[pid]["konteks"]} {passage_by_id[pid]["teks"]}'.casefold()
        for pid in soal["passage_id"]
    }
    kutipan = soal["bukti"].casefold()
    assert any(kutipan in teks for teks in teks_bukti.values()), soal["id_pertanyaan"]
    for pid, kutipan in soal.get("bukti_per_passage", {}).items():
        assert kutipan.casefold() in teks_bukti[pid], soal["id_pertanyaan"]

hasil_pertanyaan = [
    {**soal, "query": proses_pertanyaan(soal["pertanyaan"])}
    for soal in pertanyaan
]
passage_bertoken = [
    {
        "passage_id": p["id"],
        "dok_id": p["dok_id"],
        "halaman": p["halaman"],
        "token_pencarian": preprocess_passage(p),
        "versi_preprocess": VERSI_PREPROCESS,
    }
    for p in passages
]

pd.json_normalize(hasil_pertanyaan)[
    ["id_pertanyaan", "pertanyaan", "query.tipe_jawaban", "query.token_pencarian", "query.status"]
]

,id_pertanyaan,pertanyaan,query.tipe_jawaban,query.token_pencarian,query.status
0,A2-001,Berapa sks minimal untuk menyelesaikan studi s...,ANGKA,"[sks, minimal, menyelesaikan, studi, sarjana]",SIAP_RETRIEVAL
1,A2-002,Berapa lama waktu maksimal untuk Ujian Skripsi?,DURASI,"[lama, waktu, maksimal, ujian, skripsi]",SIAP_RETRIEVAL
2,A2-003,Apa itu skripsi?,DEFINISI,[skripsi],SIAP_RETRIEVAL
3,A2-004,Apa nilai huruf untuk nilai angka 78?,NILAI_HURUF,"[nilai, huruf, nilai, angka, 78]",SIAP_RETRIEVAL
4,A3-001,Berapa lama waktu maksimal untuk menyelesaikan...,DURASI,"[lama, waktu, maksimal, menyelesaikan, studi, ...",SIAP_RETRIEVAL
5,A3-002,Kapan batas pengajuan cuti akademik setelah da...,WAKTU,"[batas, pengajuan, cuti, akademik, setelah, da...",SIAP_RETRIEVAL
6,A3-003,Apa rumus indeks prestasi semester?,RUMUS,"[rumus, ip, semester]",SIAP_RETRIEVAL
7,A4-001,Berapa persentase kehadiran minimal agar mahas...,PERSENTASE,"[persentase, kehadiran, minimal, agar, mahasis...",SIAP_RETRIEVAL
8,A4-002,Berapa menit batas maksimal keterlambatan meng...,DURASI,"[menit, batas, maksimal, keterlambatan, mengik...",SIAP_RETRIEVAL
9,A4-003,Berapa bobot nilai huruf B+?,ANGKA,"[bobot, nilai, huruf, b+]",SIAP_RETRIEVAL


Tabel menampilkan tipe jawaban, token pencarian, dan status setiap soal. Pertanyaan serta passage memakai aturan normalisasi yang sama.

# 3. Simpan Hasil

In [3]:
Path("output").mkdir(exist_ok=True)
with open("output/hasil_pertanyaan.json", "w", encoding="utf-8") as file:
    json.dump(hasil_pertanyaan, file, ensure_ascii=False, indent=2)
    file.write("\n")
with open("output/passages_preprocessed.jsonl", "w", encoding="utf-8") as file:
    for passage in passage_bertoken:
        file.write(json.dumps(passage, ensure_ascii=False) + "\n")

Ekspor menyimpan soal yang sudah diproses dan token passage agar tahap retrieval dapat membangun indeks tanpa mengulang preprocessing.